In [1]:
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import torch
import cv2
from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import Counter

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from transformers import VideoMAEModel, VideoMAEImageProcessor

In [2]:
# ──────────────────────────────────────────────
# PATHS
# ──────────────────────────────────────────────

XD_TRAIN  = "/kaggle/input/datasets/bypktt/xd-violence/train"
XD_TEST   = "/kaggle/input/datasets/bypktt/xd-violence/test"

RWF_TRAIN = "/kaggle/input/datasets/vulamnguyen/rwf2000/RWF-2000/train"
RWF_TEST  = "/kaggle/input/datasets/vulamnguyen/rwf2000/RWF-2000/val"

# Where to save the index CSVs
OUTPUT_DIR = "/kaggle/working"

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────

# Classes to SKIP from XD-Violence (irrelevant for mall surveillance)
XD_SKIP_CLASSES = {"caraccident", "explosion"}

# Normal class names per dataset (lowercase)
XD_NORMAL_LABEL  = "normal"
RWF_NORMAL_LABEL = "nonfight"

## Helpers

In [3]:
def scan_dataset(root_dir, normal_label, skip_classes=None, dataset_name=""):
    """
    Scans a dataset folder and returns a list of (filepath, label, class_name, dataset).
    label: 0 = Normal, 1 = Violence
    Skips any class in skip_classes (case-insensitive).
    """
    records = []
    skip_classes = {c.lower() for c in skip_classes} if skip_classes else set()

    if not os.path.exists(root_dir):
        print(f"  WARNING: Path not found: {root_dir}")
        return records

    classes = sorted(os.listdir(root_dir))

    for class_name in classes:
        class_path = os.path.join(root_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        if class_name.lower() in skip_classes:
            print(f"  Skipping class: {class_name} ({dataset_name})")
            continue

        files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".mp4", ".avi", ".mkv", ".mov"))
        ]

        label = 0 if class_name.lower() == normal_label.lower() else 1

        for fname in files:
            records.append({
                "filepath":    os.path.join(class_path, fname),
                "label":       label,
                "class_name":  class_name,
                "dataset":     dataset_name,
            })

    return records


def print_summary(df, split_name):
    """Prints a detailed breakdown of a dataset split."""
    print(f"\n{'='*55}")
    print(f"  {split_name}")
    print(f"{'='*55}")

    total    = len(df)
    normal   = (df["label"] == 0).sum()
    violence = (df["label"] == 1).sum()

    print(f"  Total videos : {total}")
    print(f"  Normal       : {normal}  ({normal/total*100:.1f}%)")
    print(f"  Violence     : {violence}  ({violence/total*100:.1f}%)")

    print(f"\n  Breakdown by class:")
    for (dataset, class_name), grp in df.groupby(["dataset", "class_name"]):
        label_str = "Normal" if grp["label"].iloc[0] == 0 else "Violence"
        print(f"    [{dataset}]  {class_name:<18} {len(grp):>5} files  →  {label_str}")

    print(f"{'='*55}")


def build_sampler(labels):
    """
    Builds a WeightedRandomSampler that balances Normal vs Violence per batch.
    Does NOT remove any samples — just adjusts sampling probability.
    """
    label_counts = Counter(labels)
    total        = len(labels)

    # Weight for each sample = 1 / count of its class
    class_weights = {cls: total / count for cls, count in label_counts.items()}
    sample_weights = [class_weights[l] for l in labels]

    sampler = WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float),
        num_samples = total,
        replacement = True,
    )

    print(f"\n  Sampler class weights:")
    for cls, w in class_weights.items():
        label_str = "Normal" if cls == 0 else "Violence"
        print(f"    {label_str}: {w:.4f}")

    return sampler

## BUILD TRAIN SPLIT

In [4]:
print("\nScanning datasets...")
print("\n[TRAIN]")

xd_train_records  = scan_dataset(
    XD_TRAIN,
    normal_label = XD_NORMAL_LABEL,
    skip_classes = XD_SKIP_CLASSES,
    dataset_name = "XD-Violence",
)

rwf_train_records = scan_dataset(
    RWF_TRAIN,
    normal_label = RWF_NORMAL_LABEL,
    skip_classes = None,
    dataset_name = "RWF-2000",
)

train_df = pd.DataFrame(xd_train_records + rwf_train_records)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle order


Scanning datasets...

[TRAIN]
  Skipping class: CarAccident (XD-Violence)
  Skipping class: Explosion (XD-Violence)


## BUILD TEST SPLIT

In [5]:
print("\n[TEST]")

xd_test_records  = scan_dataset(
    XD_TEST,
    normal_label = XD_NORMAL_LABEL,
    skip_classes = XD_SKIP_CLASSES,
    dataset_name = "XD-Violence",
)

rwf_test_records = scan_dataset(
    RWF_TEST,
    normal_label = RWF_NORMAL_LABEL,
    skip_classes = None,
    dataset_name = "RWF-2000",
)

test_df = pd.DataFrame(xd_test_records + rwf_test_records)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)


[TEST]
  Skipping class: CarAccident (XD-Violence)
  Skipping class: Explosion (XD-Violence)


## Summaries, Sampler, And Index csv

In [6]:
# ──────────────────────────────────────────────
# PRINT SUMMARIES
# ──────────────────────────────────────────────

print_summary(train_df, "TRAIN SPLIT")
print_summary(test_df,  "TEST SPLIT")

# ──────────────────────────────────────────────
# BUILD SAMPLER FOR TRAINING
# ──────────────────────────────────────────────

print("\n[WeightedRandomSampler — Train]")
sampler = build_sampler(train_df["label"].tolist())

# ──────────────────────────────────────────────
# SAVE INDEX CSVs
# ──────────────────────────────────────────────

train_csv = os.path.join(OUTPUT_DIR, "train_index.csv")
test_csv  = os.path.join(OUTPUT_DIR, "test_index.csv")

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv,   index=False)

print(f"\n  Saved train index → {train_csv}")
print(f"  Saved test  index → {test_csv}")


  TRAIN SPLIT
  Total videos : 4663
  Normal       : 2849  (61.1%)
  Violence     : 1814  (38.9%)

  Breakdown by class:
    [RWF-2000]  Fight                800 files  →  Violence
    [RWF-2000]  NonFight             800 files  →  Normal
    [XD-Violence]  Abuse                 31 files  →  Violence
    [XD-Violence]  Fighting             377 files  →  Violence
    [XD-Violence]  Normal              2049 files  →  Normal
    [XD-Violence]  Riot                 374 files  →  Violence
    [XD-Violence]  Shooting             232 files  →  Violence

  TEST SPLIT
  Total videos : 975
  Normal       : 500  (51.3%)
  Violence     : 475  (48.7%)

  Breakdown by class:
    [RWF-2000]  Fight                200 files  →  Violence
    [RWF-2000]  NonFight             200 files  →  Normal
    [XD-Violence]  Abuse                  7 files  →  Violence
    [XD-Violence]  Fighting             107 files  →  Violence
    [XD-Violence]  Normal               300 files  →  Normal
    [XD-Violence]  Riot 

## FINAL BALANCE CHECK

In [7]:
print("\n[Final Balance Check]")
train_normal   = (train_df["label"] == 0).sum()
train_violence = (train_df["label"] == 1).sum()
test_normal    = (test_df["label"] == 0).sum()
test_violence  = (test_df["label"] == 1).sum()

print(f"  Train → Normal: {train_normal}  Violence: {train_violence}  "
      f"Ratio: {train_normal/(train_normal+train_violence)*100:.1f}/{train_violence/(train_normal+train_violence)*100:.1f}")
print(f"  Test  → Normal: {test_normal}   Violence: {test_violence}   "
      f"Ratio: {test_normal/(test_normal+test_violence)*100:.1f}/{test_violence/(test_normal+test_violence)*100:.1f}")

print("\nDataset preparation complete.")


[Final Balance Check]
  Train → Normal: 2849  Violence: 1814  Ratio: 61.1/38.9
  Test  → Normal: 500   Violence: 475   Ratio: 51.3/48.7

Dataset preparation complete.


---

---

# Model preparation and training

## Config

In [8]:
# CSV index files produced by prepare_dataset.py
TRAIN_CSV = "/kaggle/working/train_index.csv"
TEST_CSV  = "/kaggle/working/test_index.csv"

IMG_SIZE   = 224
NUM_FRAMES = 16

BATCH_SIZE = 4
EPOCHS     = 20

LR_HEAD      = 3e-4   # classifier head LR (frozen phase)
LR_BACKBONE  = 5e-6   # backbone LR (after unfreeze)
UNFREEZE_EPOCH = 7    # freeze backbone for first N epochs, then unfreeze

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "MCG-NJU/videomae-base"

# ── Checkpoint paths ──────────────────────────
BEST_MODEL_SAVE_PATH  = "/kaggle/working/best_model_combined.pth"
LATEST_CKPT_SAVE_PATH = "/kaggle/working/latest_checkpoint_combined.pth"

# Set to None for fresh start.
# Set to checkpoint path to resume a previous session.
RESUME_CHECKPOINT_PATH = None
# RESUME_CHECKPOINT_PATH = "/kaggle/input/videomae-checkpoints-combined/latest_checkpoint_combined.pth"

EARLY_STOPPING_PATIENCE = 5

## PROCESSOR

In [9]:
processor = VideoMAEImageProcessor.from_pretrained(MODEL_NAME)

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

## DATASET

In [10]:
class CombinedVideoDataset(Dataset):
    """
    Loads videos from a CSV index built by prepare_dataset.py.
    CSV columns: filepath, label, class_name, dataset
    """

    def __init__(self, csv_path):
        self.df      = pd.read_csv(csv_path)
        self.samples = self.df["filepath"].tolist()
        self.labels  = self.df["label"].tolist()

        print(f"  Loaded {len(self.samples)} videos from {csv_path}")
        counts = Counter(self.labels)
        print(f"  Normal: {counts[0]}  Violence: {counts[1]}")

    def load_video(self, path):
        cap          = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames       = []

        if total_frames <= 0:
            cap.release()
            return [
                np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                for _ in range(NUM_FRAMES)
            ]

        if total_frames < NUM_FRAMES:
            indices  = list(range(total_frames))
            indices += [total_frames - 1] * (NUM_FRAMES - total_frames)
        else:
            indices = np.linspace(0, total_frames - 1, NUM_FRAMES).astype(int)

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()

            if not ret:
                frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()
        return frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.load_video(self.samples[idx]), self.labels[idx]


def collate_fn(batch):
    videos = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    inputs = processor(videos, return_tensors="pt")
    return inputs["pixel_values"], torch.tensor(labels)


def build_sampler(labels):
    """WeightedRandomSampler — balances Normal/Violence per batch."""
    counts         = Counter(labels)
    total          = len(labels)
    class_weights  = {cls: total / count for cls, count in counts.items()}
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float),
        num_samples = total,
        replacement = True,
    )

## MODEL

In [11]:
class VideoMAEClassifier(nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone  = VideoMAEModel.from_pretrained(MODEL_NAME)
        hidden_size    = self.backbone.config.hidden_size  # 768 for base

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.5),    # slightly higher than before to reduce overfitting
            nn.Linear(256, 2),
        )

    def forward(self, pixel_values):
        outputs   = self.backbone(pixel_values=pixel_values)
        embedding = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(embedding)

## OPTIMIZER / SCHEDULER HELPERS

In [12]:
def build_optimizer_and_scheduler(model, backbone_unfrozen, epochs_remaining):
    if backbone_unfrozen:
        optimizer = optim.AdamW([
            {"params": model.backbone.parameters(),
             "lr": LR_BACKBONE, "weight_decay": 0.01},
            {"params": model.classifier.parameters(),
             "lr": LR_HEAD / 3, "weight_decay": 0.001},
        ])
    else:
        optimizer = optim.AdamW(
            model.classifier.parameters(),
            lr=LR_HEAD,
            weight_decay=0.001,
        )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs_remaining, 1), eta_min=1e-6
    )
    return optimizer, scheduler

## CHECKPOINT HELPERS

In [13]:
def save_checkpoint(path, epoch, model, optimizer, scheduler,
                    best_f1, patience, backbone_unfrozen):
    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_f1":              best_f1,
        "patience":             patience,
        "backbone_unfrozen":    backbone_unfrozen,
    }, path)
    print(f"  >> Checkpoint saved → {path}")


def load_checkpoint(path, model, device):
    print(f"\nResuming from: {path}")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return ckpt

## TRAINING

In [14]:
def train():
    print(f"\nUsing device: {DEVICE}")

    # ── Datasets ────────────────────────────────
    print("\nLoading datasets...")
    train_dataset = CombinedVideoDataset(TRAIN_CSV)
    test_dataset  = CombinedVideoDataset(TEST_CSV)

    sampler = build_sampler(train_dataset.labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size  = BATCH_SIZE,
        sampler     = sampler,       # replaces shuffle=True
        collate_fn  = collate_fn,
        num_workers = 4,
        pin_memory  = True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        collate_fn  = collate_fn,
        num_workers = 4,
        pin_memory  = True
    )

    # ── Model ───────────────────────────────────
    model = VideoMAEClassifier().to(DEVICE)

    # ── Defaults (fresh start) ───────────────────
    START_EPOCH       = 0
    best_f1           = 0.0
    patience          = 0
    backbone_unfrozen = False

    # Freeze backbone by default for phase 1
    for param in model.backbone.parameters():
        param.requires_grad = False

    optimizer, scheduler = build_optimizer_and_scheduler(
        model,
        backbone_unfrozen = False,
        epochs_remaining  = EPOCHS,
    )

    # ── Resume from checkpoint ───────────────────
    if RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
        ckpt = load_checkpoint(RESUME_CHECKPOINT_PATH, model, DEVICE)

        START_EPOCH       = ckpt["epoch"] + 1
        best_f1           = ckpt["best_f1"]
        patience          = ckpt["patience"]
        backbone_unfrozen = ckpt.get("backbone_unfrozen", False)

        if backbone_unfrozen:
            for param in model.backbone.parameters():
                param.requires_grad = True

        optimizer, scheduler = build_optimizer_and_scheduler(
            model,
            backbone_unfrozen,
            epochs_remaining = EPOCHS - START_EPOCH,
        )
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        print(f"Resumed at epoch {START_EPOCH + 1}/{EPOCHS}")
        print(f"Best F1 so far  : {best_f1:.4f}")
        print(f"Patience        : {patience}/{EARLY_STOPPING_PATIENCE}")
        print(f"Backbone frozen : {not backbone_unfrozen}")
    else:
        print("No checkpoint found — starting fresh.")

    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler()

    # ── Epoch loop ──────────────────────────────
    for epoch in range(START_EPOCH, EPOCHS):

        # Unfreeze backbone when the time comes
        if not backbone_unfrozen and epoch >= UNFREEZE_EPOCH:
            print(f"\n>>> Unfreezing backbone at epoch {epoch + 1}")
            for param in model.backbone.parameters():
                param.requires_grad = True
            backbone_unfrozen = True
            optimizer, scheduler = build_optimizer_and_scheduler(
                model,
                backbone_unfrozen = True,
                epochs_remaining  = EPOCHS - epoch,
            )

        print(f"\nEpoch {epoch + 1}/{EPOCHS}  "
              f"[backbone {'unfrozen' if backbone_unfrozen else 'frozen'}]")

        # ── Train ───────────────────────────────
        model.train()
        train_loss = 0.0

        for videos, labels in tqdm(train_loader, desc="Training"):
            videos = videos.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(videos)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        scheduler.step()

        # ── Evaluate ────────────────────────────
        model.eval()
        epoch_preds  = []
        epoch_labels = []

        with torch.no_grad():
            for videos, labels in tqdm(test_loader, desc="Evaluating"):
                videos = videos.to(DEVICE)
                with torch.cuda.amp.autocast():
                    outputs = model(videos)
                preds = outputs.argmax(dim=1)
                epoch_preds.extend(preds.cpu().numpy())
                epoch_labels.extend(labels.numpy())

        acc       = accuracy_score(epoch_labels, epoch_preds)
        precision = precision_score(epoch_labels, epoch_preds, zero_division=0)
        recall    = recall_score(epoch_labels, epoch_preds, zero_division=0)
        f1        = f1_score(epoch_labels, epoch_preds, zero_division=0)
        cm        = confusion_matrix(epoch_labels, epoch_preds)
        avg_loss  = train_loss / len(train_loader)

        print(f"\nLoss      : {avg_loss:.4f}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Recall    : {recall:.4f}")
        print(f"F1        : {f1:.4f}")
        print(f"Confusion Matrix:\n{cm}")

        # ── Always save latest checkpoint ───────
        save_checkpoint(
            LATEST_CKPT_SAVE_PATH,
            epoch, model, optimizer, scheduler,
            best_f1, patience, backbone_unfrozen,
        )

        # ── Save best model ─────────────────────
        if f1 > best_f1:
            best_f1  = f1
            patience = 0
            torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
            print(f"  >> Best model saved (F1={best_f1:.4f}) → {BEST_MODEL_SAVE_PATH}")
        else:
            patience += 1
            print(f"  No improvement. Patience: {patience}/{EARLY_STOPPING_PATIENCE}")
            if patience >= EARLY_STOPPING_PATIENCE:
                print("\nEarly stopping triggered.")
                break

    # ── Final evaluation ────────────────────────
    print("\n" + "="*55)
    print("BEST MODEL — FINAL EVALUATION")
    print("="*55)

    model.load_state_dict(torch.load(BEST_MODEL_SAVE_PATH, map_location=DEVICE))
    model.eval()

    final_preds  = []
    final_labels = []

    with torch.no_grad():
        for videos, labels in tqdm(test_loader, desc="Final Eval"):
            videos = videos.to(DEVICE)
            with torch.cuda.amp.autocast():
                outputs = model(videos)
            preds = outputs.argmax(dim=1)
            final_preds.extend(preds.cpu().numpy())
            final_labels.extend(labels.numpy())

    print(f"Accuracy  : {accuracy_score(final_labels, final_preds):.4f}")
    print(f"Precision : {precision_score(final_labels, final_preds, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(final_labels, final_preds, zero_division=0):.4f}")
    print(f"F1        : {f1_score(final_labels, final_preds, zero_division=0):.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(final_labels, final_preds)}")

## INFERENCE

In [15]:
def extract_frames(path, num_frames=NUM_FRAMES, img_size=IMG_SIZE):
    cap          = cv2.VideoCapture(path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames       = []

    if total_frames <= 0:
        cap.release()
        return [np.zeros((img_size, img_size, 3), dtype=np.uint8)
                for _ in range(num_frames)]

    if total_frames < num_frames:
        indices  = list(range(total_frames))
        indices += [total_frames - 1] * (num_frames - total_frames)
    else:
        indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames


def load_model_for_inference(weights_path=BEST_MODEL_SAVE_PATH):
    model = VideoMAEClassifier()
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("Model loaded successfully.")
    return model


def predict_video(video_path, model, threshold=0.4):
    """
    threshold: violence confidence needed to trigger alert.
               Lower = more sensitive (higher recall, more false alarms).
               0.4 is a good starting point for a security system.
    """
    print(f"\nProcessing: {video_path}")

    frames       = extract_frames(video_path)
    inputs       = processor([frames], return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)

    with torch.no_grad():
        logits = model(pixel_values=pixel_values)
        probs  = torch.softmax(logits, dim=1)[0]

    violence_conf = probs[1].item()
    prediction    = "Violence / Anomaly" if violence_conf >= threshold else "Normal"

    print(f"Confidence : Normal={probs[0]:.2%}  Violence={probs[1]:.2%}")
    print(f"Threshold  : {threshold:.0%}")
    print(f"Prediction : --> {prediction} <--")
    return prediction, probs.cpu().numpy()

## ENTRY POINT(TRAINING)

In [16]:
if __name__ == "__main__":

    train()


Using device: cuda

Loading datasets...
  Loaded 4663 videos from /kaggle/working/train_index.csv
  Normal: 2849  Violence: 1814
  Loaded 975 videos from /kaggle/working/test_index.csv
  Normal: 500  Violence: 475


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias    

...


Loss      : 0.0777
Accuracy  : 0.8646
Precision : 0.8493
Recall    : 0.8779
F1        : 0.8634
Confusion Matrix:
[[426  74]
 [ 58 417]]
  >> Checkpoint saved → /kaggle/working/latest_checkpoint_combined.pth
  >> Best model saved (F1=0.8634) → /kaggle/working/best_model_combined.pth

Epoch 15/20  [backbone unfrozen]


Training:   0%|          | 0/1166 [00:00<?, ?it/s]/tmp/ipykernel_58/1762027319.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training:  16%|█▌        | 185/1166 [05:50<16:28,  1.01s/it][mov,mp4,m4a,3gp,3g2,mj2 @ 0x6bbd0640] Referenced QT chapter track not found
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x6bc04800] Referenced QT chapter track not found
Evaluating:   0%|          | 0/244 [00:00<?, ?it/s]/tmp/ipykernel_58/1762027319.py:128: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Evaluating: 100%|██████████| 244/244 [07:26<00:00,  1.83s/it]



Loss      : 0.0276
Accuracy  : 0.8564
Precision : 0.9036
Recall    : 0.7895
F1        : 0.8427
Confusion Matrix:
[[460  40]
 [100 375]]
  >> Checkpoint saved → /kaggle/working/latest_checkpoint_combined.pth
  No improvement. Patience: 1/5

Epoch 16/20  [backbone unfrozen]


Training:   0%|          | 0/1166 [00:00<?, ?it/s]/tmp/ipykernel_58/1762027319.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training:  78%|███████▊  | 905/1166 [29:28<08:29,  1.95s/it]


KeyboardInterrupt: 

## ENTRY POINT(INFERENCE)

In [ ]:
# 

## ZIPPING MODELS

In [17]:
!zip outputs.zip best_model_combined.pth latest_checkpoint_combined.pth

  adding: best_model_combined.pth (deflated 7%)
  adding: latest_checkpoint_combined.pth (deflated 8%)
